# Evals dashboard — Niche Research Agent

Интерактивный запуск component + system evals и визуализация результатов.

**Требования:**
- Ollama запущена на хосте, модели подтянуты (`qwen2.5:3b-instruct-q4_K_M`, `nomic-embed-text`).
- (Опционально) ChromaDB поднят (`docker compose up -d chromadb`) — для RAG-метрик.
- KB ингестнута: `python -m src.memory.semantic ingest knowledge_base/`.

Этот ноутбук — обёртка над CLI-скриптами `component_evals.py` и `system_evals.py`. Они пишут JSON в `evals/last_*_results.json` — отсюда мы их и читаем.

In [ ]:
import json, sys, subprocess
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt

PROJECT_ROOT = Path('..').resolve()
EVALS_DIR = PROJECT_ROOT / 'evals'
sys.path.insert(0, str(PROJECT_ROOT))
print('Project:', PROJECT_ROOT)

## 1. Component evals

Прогоняем изолированные тесты: specs normalizer, USP classifier, PRD validator, Scout JSON parse rate.

In [ ]:
# Запускаем component evals (можно закомментировать если уже есть результаты)
subprocess.run([sys.executable, '-m', 'evals.component_evals'], cwd=PROJECT_ROOT, check=False)

In [ ]:
comp = json.loads((EVALS_DIR / 'last_component_results.json').read_text(encoding='utf-8'))
comp.keys()

In [ ]:
rows = []
if 'specs_normalizer' in comp:
    rows.append(('specs_normalizer', 'accuracy', comp['specs_normalizer']['accuracy'], 0.85))
if 'usp_classifier' in comp:
    rows.append(('usp_classifier', 'accuracy', comp['usp_classifier']['accuracy'], 0.70))
if 'scout_parse' in comp and 'parse_rate' in comp['scout_parse']:
    rows.append(('scout', 'parse_rate', comp['scout_parse']['parse_rate'], 0.95))
    rows.append(('scout', 'category_match', comp['scout_parse']['category_match_rate'], 0.80))

df_comp = pd.DataFrame(rows, columns=['component', 'metric', 'value', 'threshold'])
df_comp['passed'] = df_comp['value'] >= df_comp['threshold']
df_comp

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4))
labels = df_comp['component'] + ' / ' + df_comp['metric']
colors = ['#3a9d23' if p else '#c43c3c' for p in df_comp['passed']]
ax.barh(labels, df_comp['value'], color=colors)
for i, t in enumerate(df_comp['threshold']):
    ax.axvline(t, ymin=(i)/len(df_comp), ymax=(i+1)/len(df_comp), color='black', linestyle='--', linewidth=1)
ax.set_xlim(0, 1.05)
ax.set_xlabel('Value')
ax.set_title('Component eval metrics (bar — actual, dashed — threshold)')
plt.tight_layout()
plt.show()

## 2. System evals

Полный pipeline на эталонных запросах из `data/eval_queries.json`. Долго — ожидайте ~30-60 сек на запрос.

In [ ]:
# Можно запустить только подмножество, прокинув ID запросов
# subprocess.run([sys.executable, '-m', 'evals.system_evals', 'q01_toothbrush_basic', 'q03_dryer_basic'], cwd=PROJECT_ROOT, check=False)
subprocess.run([sys.executable, '-m', 'evals.system_evals'], cwd=PROJECT_ROOT, check=False)

In [ ]:
sys_res = json.loads((EVALS_DIR / 'last_system_results.json').read_text(encoding='utf-8'))
df_sys = pd.DataFrame(sys_res['results'])
df_sys['passed'] = df_sys['checks'].apply(lambda c: (c or {}).get('passed', False))
df_sys[['id', 'verdict', 'judge_score', 'elapsed_s', 'passed']]

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(14, 4))

# Pass rate
pass_counts = df_sys['passed'].value_counts()
axes[0].pie(pass_counts.values, labels=['passed' if v else 'failed' for v in pass_counts.index], autopct='%1.0f%%', colors=['#3a9d23', '#c43c3c'])
axes[0].set_title(f'Task success rate = {sys_res["task_success_rate"]}')

# Verdict distribution
vd = df_sys['verdict'].fillna('none').value_counts()
axes[1].bar(vd.index, vd.values, color='#3a6ea0')
axes[1].set_title('Verdict distribution')
axes[1].tick_params(axis='x', rotation=20)

# Judge score histogram
scores = df_sys['judge_score'].dropna()
if len(scores):
    axes[2].hist(scores, bins=range(1, 7), edgecolor='black', color='#a06b3a')
    axes[2].set_xticks(range(1, 6))
    axes[2].set_title(f'Judge score (avg={sys_res.get("avg_judge_score")})')
else:
    axes[2].set_title('Judge score (no data)')

plt.tight_layout()
plt.show()

In [ ]:
# Latency per query
fig, ax = plt.subplots(figsize=(10, 4))
ax.bar(df_sys['id'], df_sys['elapsed_s'], color='#888')
ax.set_ylabel('Latency, sec')
ax.set_title(f'E2E latency per eval query (p95 ≤ 300 sec target)')
ax.axhline(300, color='red', linestyle='--', label='p95 target')
ax.tick_params(axis='x', rotation=70)
ax.legend()
plt.tight_layout()
plt.show()

## 3. Failure breakdown

Смотрим конкретно где упало.

In [ ]:
failed = df_sys[df_sys['passed'] == False]
for _, row in failed.iterrows():
    print(f"--- {row['id']}: {row['query']}")
    print(f"    verdict: {row['verdict']}")
    print(f"    checks:  {row['checks']}")
    print(f"    judge:   {row['judge_score']} — {row.get('judge_rationale', '')[:150]}")
    print()

## 4. RAG sanity check (опционально)

Проверяем что KB ингестнута и retrieve возвращает осмысленные чанки.

In [ ]:
from src.memory.semantic import get_semantic_memory
mem = get_semantic_memory()
print('health:', mem.health())

test_queries = [
    'комиссия WB для фенов',
    'требования к фото карточки',
    'обязательные документы для эпилятора',
    'must-have характеристики зубных щёток',
]
for q in test_queries:
    print(f'\n>>> {q}')
    for ch in mem.retrieve(q, k=3):
        print(f'  {ch.score:.3f}  {ch.source}')